<a href="https://colab.research.google.com/github/Toran98/111/blob/main/ZooTag_Group_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import zipfile
import os

drive.mount('/content/drive')
zip_path = '/content/drive/MyDrive/DATASET.zip'
extract_to = '/content/raw_data'

os.makedirs(extract_to, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print("Unzipped! Contents inside /content/raw_data:")
print(os.listdir(extract_to))

In [ ]:
!pip install -q pillow-heif

import os
import random
from PIL import Image
from pillow_heif import register_heif_opener

register_heif_opener()
SOURCE_DATASET_DIR = '/content/raw_data/DATASET'
OUTPUT_DIR = '/content/ZooTag_Clean_Data'
TRAIN_DIR = os.path.join(OUTPUT_DIR, 'TRAINING')
TEST_DIR = os.path.join(OUTPUT_DIR, 'TESTING')

REQUIRED_LABELS = [
    'AMPHIBIAN',
    'BIRD',
    'FISH',
    'INSECT',
    'MAMMAL',
    'MARINE_INVERTEBRATE',
    'REPTILE'
]

SPLIT_RATIO = 0.80
random.seed(42)
VALID_EXTS = ('.jpg', '.jpeg', '.png', '.heic', '.heif', '.webp', '.bmp', '.tiff')

summary = {}

for label in REQUIRED_LABELS:
    src_folder = os.path.join(SOURCE_DATASET_DIR, label)
    train_label_dir = os.path.join(TRAIN_DIR, label)
    test_label_dir = os.path.join(TEST_DIR, label)

    os.makedirs(train_label_dir, exist_ok=True)
    os.makedirs(test_label_dir, exist_ok=True)

    if not os.path.exists(src_folder):
        print(f" Folder not found: {src_folder} (Check folder spelling!)")
        continue

    all_files = [
        f for f in os.listdir(src_folder)
        if os.path.isfile(os.path.join(src_folder, f))
        and not f.startswith('.')
        and f.lower().endswith(VALID_EXTS)
    ]

    random.shuffle(all_files)
    split_idx = int(len(all_files) * SPLIT_RATIO)

    train_files = all_files[:split_idx]
    test_files = all_files[split_idx:]

    def process_and_save(file_list, target_dir, prefix):
        saved_count = 0
        for idx, fname in enumerate(file_list):
            fpath = os.path.join(src_folder, fname)
            try:
                with Image.open(fpath) as img:
                    rgb_img = img.convert('RGB')
                    rgb_img.thumbnail((1024, 1024))
                    out_name = f"{label}_{prefix}_{idx:04d}.jpg"
                    rgb_img.save(os.path.join(target_dir, out_name), 'JPEG', quality=95)
                    saved_count += 1
            except Exception as e:
                print(f" Skipped unreadable file {fname} in {label}: {e}")
        return saved_count

    n_train = process_and_save(train_files, train_label_dir, 'train')
    n_test = process_and_save(test_files, test_label_dir, 'test')
    summary[label] = (n_train, n_test)

print("\n FINAL DATASET SUMMARY (Clean RGB .JPGs):")
print("-" * 50)
for label, (tr, te) in summary.items():
    print(f"{label:22s} | TRAIN: {tr:4d} | TEST: {te:4d} | TOTAL: {tr+te:4d}")
print("-" * 50)

In [ ]:
import os
import random
from PIL import Image, ImageEnhance, ImageOps

TRAIN_DIR = '/content/ZooTag_Clean_Data/TRAINING'
TARGET_MIN_COUNT = 200  # Bring minority classes up to 200 training images
random.seed(42)

def augment_image(img):
    # 1. Random horizontal flip
    if random.random() > 0.5:
        img = ImageOps.mirror(img)
    # 2. Random slight rotation (-20 to +20 degrees)
    angle = random.uniform(-20, 20)
    img = img.rotate(angle, resample=Image.BILINEAR, expand=False)
    # 3. Random slight zoom/crop (85% to 100% of image)
    w, h = img.size
    crop_scale = random.uniform(0.85, 0.98)
    new_w, new_h = int(w * crop_scale), int(h * crop_scale)
    left = random.randint(0, w - new_w)
    top = random.randint(0, h - new_h)
    img = img.crop((left, top, left + new_w, top + new_h)).resize((w, h), Image.BILINEAR)
    # 4. Random brightness & contrast (simulating museum lighting/glass)
    img = ImageEnhance.Brightness(img).enhance(random.uniform(0.8, 1.2))
    img = ImageEnhance.Contrast(img).enhance(random.uniform(0.85, 1.15))
    return img

print("Balancing TRAINING folders to at least 200 images per class...\n")

for label in sorted(os.listdir(TRAIN_DIR)):
    label_dir = os.path.join(TRAIN_DIR, label)
    if not os.path.isdir(label_dir):
        continue

    # Only use original training files as base for augmentation
    orig_files = [f for f in os.listdir(label_dir) if '_aug_' not in f and f.endswith('.jpg')]
    current_all = [f for f in os.listdir(label_dir) if f.endswith('.jpg')]

    needed = TARGET_MIN_COUNT - len(current_all)
    if needed > 0:
        for i in range(needed):
            src_file = random.choice(orig_files)
            with Image.open(os.path.join(label_dir, src_file)) as img:
                aug_img = augment_image(img)
                aug_name = f"{label}_aug_{i:04d}.jpg"
                aug_img.save(os.path.join(label_dir, aug_name), 'JPEG', quality=95)

    final_count = len([f for f in os.listdir(label_dir) if f.endswith('.jpg')])
    print(f" {label:22s} | Original: {len(orig_files):3d} -> Balanced TRAIN: {final_count:3d}")

In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

TRAIN_DIR = '/content/ZooTag_Clean_Data/TRAINING'
TEST_DIR = '/content/ZooTag_Clean_Data/TESTING'

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# 1. Load TRAINING dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels='inferred',
    label_mode='categorical', # One-hot encoded for 7 classes
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

# 2. Load TESTING dataset
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels='inferred',
    label_mode='categorical',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
print("\n Model Class Names (Exact Order):", class_names)

# Verify pixel range and shape
for images, labels in train_ds.take(1):
    print(f" Image Batch Shape: {images.shape} (Expected: (32, 224, 224, 3))")
    print(f" Pixel Value Range: Min={tf.reduce_min(images).numpy():.1f}, Max={tf.reduce_max(images).numpy():.1f} (Expected: 0.0 to 255.0)")

# Compute subtle class weights to handle the remaining difference between 200 and 299 (BIRD)
y_train_indices = []
for _, labels in train_ds.unbatch():
    y_train_indices.append(np.argmax(labels.numpy()))
y_train_indices = np.array(y_train_indices)

weights = compute_class_weight('balanced', classes=np.unique(y_train_indices), y=y_train_indices)
class_weight_dict = {i: round(float(w), 3) for i, w in enumerate(weights)}
print(" Class Weights:", class_weight_dict)

# Optimize pipeline speed with prefetching
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Explicit Input Layer
inputs = tf.keras.Input(shape=(224, 224, 3), name="input_image")

# 2. Data Augmentation
x = layers.RandomFlip("horizontal")(inputs)
x = layers.RandomRotation(0.08)(x)
x = layers.RandomZoom(0.1)(x)
x = layers.RandomContrast(0.1)(x)

# 3. Internal Preprocessing Layer
x = layers.Rescaling(scale=1./127.5, offset=-1.0, name="internal_rescaling")(x)

# 4. Load Pre-trained MobileNetV2 Base
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze base layers

# Pass preprocessed input through the frozen base
x = base_model(x, training=False)

# 5. Add Custom Classification Head
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(7, activation='softmax', name="predictions")(x)

model = models.Model(inputs=inputs, outputs=outputs, name="ZooTag_MobileNetV2")

# Compile for Phase 1
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)

model.summary()

In [ ]:
callbacks_phase1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-5,
        verbose=1
    )
]

print("Training the Classification Head")
history_phase1 = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=15,
    class_weight=class_weight_dict,
    callbacks=callbacks_phase1
)

In [ ]:
# Unfreeze the base model
base_model.trainable = True

# Freeze all layers EXCEPT the last 40 layers, and keep all BatchNormalization layers frozen
fine_tune_at = len(base_model.layers) - 40
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
for layer in base_model.layers[fine_tune_at:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

# Recompile with a much lower learning rate so we don't destroy pre-trained features
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)

callbacks_phase2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        '/content/zootag_best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print("Fine-Tuning Top Base Layers")
history_phase2 = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=20,
    class_weight=class_weight_dict,
    callbacks=callbacks_phase2
)

# Save final model in .h5 format as required by Part 3
model.save('/content/zootag_final_model.h5')
print("\n Model saved as /content/zootag_final_model.h5")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Collect all true labels and model predictions on the unseen TESTING set
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# 1. Print Overall Test Accuracy
overall_acc = accuracy_score(y_true, y_pred)
print(f"\n FINAL UNSEEN TEST SET ACCURACY: {overall_acc * 100:.2f}%\n")

# 2. Print Precision, Recall, and F1-Score for each of the 7 categories
print(" CLASSIFICATION REPORT (Precision, Recall, F1-Score per Class):")
print("-" * 70)
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# 3. Plot Confusion Matrix (Save this image for your Model Card!)
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title(f'ZooTag Confusion Matrix (Test Accuracy: {overall_acc*100:.2f}%)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=300)
plt.show()

In [ ]:
import tensorflow as tf
import numpy as np
import os

# 1. Build a clean inference-only wrapper
inf_inputs = tf.keras.Input(shape=(224, 224, 3), name="input_image")
x_inf = model.get_layer("internal_rescaling")(inf_inputs)
x_inf = base_model(x_inf, training=False)

# Pass through ALL 7 layers of the trained classification head
for layer in model.layers[-7:]:
    x_inf = layer(x_inf)

inference_model = tf.keras.Model(inputs=inf_inputs, outputs=x_inf, name="ZooTag_TFLite_Model")

# Save clean .h5 model as well
inference_model.save('/content/zootag_clean_inference.h5')

# 2. Convert to TensorFlow Lite (.tflite)
converter = tf.lite.TFLiteConverter.from_keras_model(inference_model)
tflite_model = converter.convert()

tflite_path = '/content/zootag_model.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
print(f" TFLite model saved to {tflite_path} (Size: {size_mb:.2f} MB)\n")

# 3. Post-Conversion Verification: Load the .tflite model and test on TESTING/ images
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(" TFLite Input Specification Check:")
print(f"   Input Shape : {input_details[0]['shape']} (Expected: [1, 224, 224, 3])")
print(f"   Input Dtype : {input_details[0]['dtype']} (Expected: float32 in [0, 255])\n")

tflite_correct = 0
keras_tflite_matches = 0
total_images = 0

for images, labels in test_ds.unbatch():
    img_tensor = tf.expand_dims(images, axis=0).numpy().astype(np.float32)
    true_label = np.argmax(labels.numpy())

    # Original Keras prediction
    keras_pred = np.argmax(model.predict(img_tensor, verbose=0)[0])

    # TFLite prediction
    interpreter.set_tensor(input_details[0]['index'], img_tensor)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])
    tflite_pred = np.argmax(tflite_output[0])

    if tflite_pred == true_label:
        tflite_correct += 1
    if tflite_pred == keras_pred:
        keras_tflite_matches += 1

    total_images += 1

tflite_acc = (tflite_correct / total_images) * 100
match_rate = (keras_tflite_matches / total_images) * 100

print("POST-CONVERSION VERIFICATION RESULTS:")
print("-" * 55)
print(f"Total Unseen Test Images Verified : {total_images}")
print(f"Original Keras Model Accuracy     : {overall_acc * 100:.2f}%")
print(f"Converted TFLite Model Accuracy   : {tflite_acc:.2f}%")
print(f"Prediction Agreement (Match Rate) : {match_rate:.2f}% ({keras_tflite_matches}/{total_images})")
print("-" * 55)